# 02 – Cleaning the quotes and estimating the forward

This notebook prepares the option quotes for the implied volatility calculation. The steps are:

1. keep only valid quotes;
2. choose one option family per expiry (SPXW or SPX);
3. compute the time to expiry in years;
4. build continuously compounded discount rates from the Treasury curve;
5. estimate the forward price of each expiry from put-call parity;
6. keep only out-of-the-money options and compute the log-moneyness.

The output is the file `SPX_2026-10-26_1030_clean.csv`.

In [1]:
import os
import numpy as np
import pandas as pd

SNAPSHOT = "SPX_2026-10-26_1030"      # the market snapshot used in the whole project
DATA_DIR = "data"                     # folder with all input and output files

df = pd.read_csv(os.path.join(DATA_DIR, SNAPSHOT + ".csv"))
print(df.shape)
print(df.columns.tolist())

(19594, 18)
['contractSymbol', 'lastTradeDate', 'strike', 'lastPrice', 'bid', 'ask', 'change', 'percentChange', 'volume', 'openInterest', 'impliedVolatility', 'inTheMoney', 'contractSize', 'currency', 'type', 'expiry', 'snapshot_time', 'spot']


## 1. Valid quotes

We use the mid price, $(\text{bid} + \text{ask}) / 2$. A quote is kept only if:

- the bid is positive, so somebody is really willing to buy;
- the ask is not below the bid;
- the mid price is above the intrinsic value computed with the spot price.

The last rule is a simple sanity check. For a European put the true lower bound is
$K e^{-rT} - S e^{-qT}$, which can be lower than $K - S$, so some deep in-the-money puts may be removed
by mistake. This has no effect on the results, because in Section 6 we keep only out-of-the-money
options, whose intrinsic value is zero or very small.

In [2]:
df["mid"] = (df["bid"] + df["ask"]) / 2

call_value = np.maximum(df["spot"] - df["strike"], 0)     # intrinsic value of a call
put_value = np.maximum(df["strike"] - df["spot"], 0)      # intrinsic value of a put
df["intrinsic"] = np.where(df["type"] == "C", call_value, put_value)

df["valid_quote"] = (df["bid"] > 0) & (df["ask"] >= df["bid"]) & (df["mid"] > df["intrinsic"])

clean = df[df["valid_quote"]].copy()
print(len(clean), "valid out of", len(df))

16829 valid out of 19594


## 2. SPXW or SPX for each expiry

Two option families are listed on the index:

- **SPXW** (weeklys and end-of-month options) settle at the closing price of the expiry day;
- **SPX** (standard monthly options) settle at the opening price of the expiry day.

When an expiry has both families, we keep only SPXW, so that each expiry has one set of quotes with
one settlement time. When only SPX options exist, we keep them.

In [3]:
clean["root"] = np.where(clean["contractSymbol"].str.startswith("SPXW"), "SPXW", "SPX")
print(clean["root"].value_counts())

expiries_with_spxw = set(clean.loc[clean["root"] == "SPXW", "expiry"])
keep = (clean["root"] == "SPXW") | (~clean["expiry"].isin(expiries_with_spxw))

clean = clean[keep].copy()
print(len(clean), "rows,", clean["expiry"].nunique(), "expiries")

root
SPXW    11192
SPX      5637
Name: count, dtype: int64
14738 rows, 54 expiries


## 3. Time to expiry

The time to expiry is measured in years, from the snapshot to the settlement time of the option
(16:00 New York time for SPXW, 9:30 for SPX), with 365 days in a year.

Expiries shorter than 7 days are removed. Their smiles are very steep and they are not needed for
hedges with a maturity of several months.

In [4]:
snapshot = pd.to_datetime(clean["snapshot_time"].iloc[0])

hour = np.where(clean["root"] == "SPXW", "16:00", "09:30")       # settlement time
expiry_time = pd.to_datetime(clean["expiry"] + " " + hour).dt.tz_localize("America/New_York")

clean["T"] = (expiry_time - snapshot).dt.total_seconds() / (365 * 24 * 3600)
clean = clean[clean["T"] >= 7 / 365].copy()

print(len(clean), "rows,", clean["expiry"].nunique(), "expiries")
print("T from", round(clean["T"].min(), 4), "to", round(clean["T"].max(), 4), "years")

13027 rows, 49 expiries
T from 0.0199 to 5.1507 years


## 4. Discount rates

Discount rates come from the US Treasury constant-maturity yields published by FRED (series DGS1MO,
DGS3MO, DGS6MO, DGS1, DGS2, DGS3 and DGS5), read on the snapshot date. The yields are annual rates,
so we convert them to continuously compounded rates with

$$r = \ln(1 + y).$$

For any maturity between two points of the curve, the rate is interpolated linearly in $T$.

In [5]:
maturities = [1/12, 1/4, 1/2, 1, 2, 3, 5]                            # years
yields = [0.0407, 0.0418, 0.0432, 0.0447, 0.0467, 0.0480, 0.0485]    # decimals, e.g. 0.0407 = 4.07%

rates_cont = np.log(1 + np.array(yields))       # continuously compounded rates
print(np.round(rates_cont, 5))

[0.03989 0.04095 0.04229 0.04373 0.04564 0.04688 0.04736]


## 5. Forward price from put-call parity

For European options with the same strike and expiry, put-call parity gives

$$C - P = e^{-rT}\,(F - K),$$

so the forward price can be read directly from option prices:

$$\boxed{F = K + \frac{C - P}{e^{-rT}}}$$

This forward already includes the dividends expected by the market, so we do not need a separate
dividend assumption. For each expiry we take the five strikes where both a call and a put are quoted
and $|C - P|$ is smallest (the strikes closest to the money), compute one forward per strike, and use
the median.

In [6]:
def forward_from_parity(K, C, P, T, r):
    D = np.exp(-r * T)          # discount factor
    return K + (C - P) / D

In [7]:
results = []

for exp, group in clean.groupby("expiry"):
    calls = group[group["type"] == "C"].set_index("strike")["mid"]
    puts = group[group["type"] == "P"].set_index("strike")["mid"]

    pairs = pd.concat({"C": calls, "P": puts}, axis=1).dropna()     # strikes with both a call and a put
    pairs["abs_diff"] = (pairs["C"] - pairs["P"]).abs()
    near = pairs.sort_values("abs_diff").head(5)                    # the 5 strikes closest to the money

    T = group["T"].iloc[0]
    r = np.interp(T, maturities, rates_cont)

    F_values = []
    for K, row in near.iterrows():
        F_values.append(forward_from_parity(K, row["C"], row["P"], T, r))

    results.append({"expiry": exp, "T": T, "r": r, "F": np.median(F_values)})

fw = pd.DataFrame(results)
print(len(fw), "expiries")
print(fw.head(10))

49 expiries
       expiry         T         r            F
0  2026-11-02  0.019919  0.039894  7677.443998
1  2026-11-03  0.022659  0.039894  7678.257461
2  2026-11-04  0.025399  0.039894  7679.354410
3  2026-11-05  0.028138  0.039894  7680.161400
4  2026-11-06  0.030878  0.039894  7682.146487
5  2026-11-09  0.039097  0.039894  7682.611863
6  2026-11-10  0.041837  0.039894  7683.781291
7  2026-11-11  0.044577  0.039894  7684.039410
8  2026-11-12  0.047316  0.039894  7684.627585
9  2026-11-13  0.050056  0.039894  7686.282609


### Example: the expiry closest to one month

The cell below shows the five pairs used for the expiry closest to one month. In the snapshot of
26 October 2026 this is the expiry of 20 November 2026. For example, at $K = 7685$ we have
$C - P = 118.70 - 115.00 = 3.70$, and with $T = 0.0692$ and $r = 3.99\%$ the discount factor is
$e^{-rT} = 0.99724$, so

$$F = 7685 + \frac{3.70}{0.99724} = 7688.71.$$

The five pairs give forwards between 7688.56 and 7689.12, and the median is **7688.71**.

**Check with the spot price.** The spot is $S = 7673.61$, so the carry implied by the forward is

$$\frac{\ln(F/S)}{T} = \frac{\ln(7688.71 / 7673.61)}{0.0692} \approx 2.8\%.$$

The carry is the interest rate minus the dividend yield. With a rate of about 4.0%, it implies a
dividend yield of about 1.2%, which is a realistic value for the S&P 500. If the forward had come
out at 7500 or 7900, this check would have shown an error in the data.

In [8]:
exp = fw.loc[(fw["T"] - 1/12).abs().idxmin(), "expiry"]      # expiry closest to 1 month
print("Expiry:", exp)
group = clean[clean["expiry"] == exp]
calls = group[group["type"] == "C"].set_index("strike")["mid"]
puts = group[group["type"] == "P"].set_index("strike")["mid"]
pairs = pd.concat({"C": calls, "P": puts}, axis=1).dropna()
pairs["abs_diff"] = (pairs["C"] - pairs["P"]).abs()
print(pairs.sort_values("abs_diff").head(5))

row = fw[fw["expiry"] == exp].iloc[0]
S = group["spot"].iloc[0]
print("T:", round(row["T"], 4), " r:", round(row["r"], 4), " F:", round(row["F"], 2), " S:", S)
print("Implied carry:", round(np.log(row["F"] / S) / row["T"] * 100, 2), "%")

Expiry: 2026-11-25
             C       P  abs_diff
strike                          
7700.0  120.35  128.55      8.20
7680.0  132.40  120.30     12.10
7675.0  135.35  118.60     16.75
7710.0  114.35  132.45     18.10
7725.0  106.20  139.50     33.30
T: 0.0829  r: 0.0399  F: 7691.81  S: 7673.61
Implied carry: 2.86 %


## 6. Out-of-the-money options and log-moneyness

For each strike we keep only the out-of-the-money option: puts with $K < F$ and calls with $K \geq F$.
Out-of-the-money options are more liquid and their price is almost all time value, so they give
the most reliable implied volatilities.

The position of a strike is measured by the log-moneyness

$$k = \ln\left(\frac{K}{F}\right),$$

which is negative for puts below the forward and positive for calls above it.

In [9]:
clean = clean.merge(fw[["expiry", "F", "r"]], on="expiry")

is_otm_put = (clean["type"] == "P") & (clean["strike"] < clean["F"])
is_otm_call = (clean["type"] == "C") & (clean["strike"] >= clean["F"])
otm = clean[is_otm_put | is_otm_call].copy()

otm["k"] = np.log(otm["strike"] / otm["F"])

print(len(otm), "OTM options")
print(otm["type"].value_counts())
print("k from", round(otm["k"].min(), 3), "to", round(otm["k"].max(), 3))

8997 OTM options
type
P    6011
C    2986
Name: count, dtype: int64
k from -3.129 to 0.74


In [10]:
out_path = os.path.join(DATA_DIR, SNAPSHOT + "_clean.csv")
otm.to_csv(out_path, index=False)
print("Saved:", out_path)

Saved: data/SPX_2026-10-26_1030_clean.csv
